In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

data = [
    (1, "C001", "Laptop", "50000", "2024-01-01"),   (2, "C002", "Mobile", None, "2024-01-02"),
   (3, "C003", "Tablet", "20000", "2024-01-03"),   (4, "C004", "Laptop", "55000", "2024-01-04"),
    (5, "C005", "Headphones", None, "2024-01-05"),
    (3, "C003", "Tablet", "22000", "2024-01-06"),     (6, "C006", "Camera", "30000", "2024-01-06"),
    (7, "C007", "Mobile", "18000", "2024-01-07"),
   (8, "C008", "Watch", "8000", "2024-01-07")]
columns = ["order_id", "customer_id", "product", "amount", "updated_date"]
df = spark.createDataFrame(data, columns)
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,null,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,null,2024-01-05
3,C003,Tablet,22000,2024-01-06
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
df=df.fillna({"amount":"1000"})
df.display()

order_id,customer_id,product,amount,updated_date
1,C001,Laptop,50000,2024-01-01
2,C002,Mobile,1000,2024-01-02
3,C003,Tablet,20000,2024-01-03
4,C004,Laptop,55000,2024-01-04
5,C005,Headphones,1000,2024-01-05
3,C003,Tablet,22000,2024-01-06
6,C006,Camera,30000,2024-01-06
7,C007,Mobile,18000,2024-01-07
8,C008,Watch,8000,2024-01-07


In [0]:
from pyspark.sql.functions import col
df=df.withColumn("amount",col("amount").cast("int"))\
    .withColumn("updated_date",col("updated_date").cast("date"))
df.show()

+--------+-----------+----------+------+------------+
|order_id|customer_id|   product|amount|updated_date|
+--------+-----------+----------+------+------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|
|       2|       C002|    Mobile|  1000|  2024-01-02|
|       3|       C003|    Tablet| 20000|  2024-01-03|
|       4|       C004|    Laptop| 55000|  2024-01-04|
|       5|       C005|Headphones|  1000|  2024-01-05|
|       3|       C003|    Tablet| 22000|  2024-01-06|
|       6|       C006|    Camera| 30000|  2024-01-06|
|       7|       C007|    Mobile| 18000|  2024-01-07|
|       8|       C008|     Watch|  8000|  2024-01-07|
+--------+-----------+----------+------+------------+



In [0]:
df.printSchema()

root
 |-- order_id: long (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- updated_date: date (nullable = true)



###Add derived columns

In [0]:
from pyspark.sql.functions import when
df=df.withColumn("bonus",col("amount")*0.1)\
  .withColumn("category", when(col("amount")>20000,"High").otherwise("Low"))
df.show()

+--------+-----------+----------+------+------------+------+--------+
|order_id|customer_id|   product|amount|updated_date| bonus|category|
+--------+-----------+----------+------+------------+------+--------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|    High|
|       2|       C002|    Mobile|  1000|  2024-01-02| 100.0|     Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|     Low|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|    High|
|       5|       C005|Headphones|  1000|  2024-01-05| 100.0|     Low|
|       3|       C003|    Tablet| 22000|  2024-01-06|2200.0|    High|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|    High|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|     Low|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|     Low|
+--------+-----------+----------+------+------------+------+--------+



###Create amount_bucket

In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "amount_bucket",
    when(col("amount") < 10000, "Low")
    .when((col("amount") >= 10000) & (col("amount") <= 30000), "Medium")
    .when(col("amount") > 30000, "High")
    .otherwise("Unknown")   
)

df.show()

+--------+-----------+----------+------+------------+------+--------+-------------+
|order_id|customer_id|   product|amount|updated_date| bonus|category|amount_bucket|
+--------+-----------+----------+------+------------+------+--------+-------------+
|       1|       C001|    Laptop| 50000|  2024-01-01|5000.0|    High|         High|
|       2|       C002|    Mobile|  1000|  2024-01-02| 100.0|     Low|          Low|
|       3|       C003|    Tablet| 20000|  2024-01-03|2000.0|     Low|       Medium|
|       4|       C004|    Laptop| 55000|  2024-01-04|5500.0|    High|         High|
|       5|       C005|Headphones|  1000|  2024-01-05| 100.0|     Low|          Low|
|       3|       C003|    Tablet| 22000|  2024-01-06|2200.0|    High|       Medium|
|       6|       C006|    Camera| 30000|  2024-01-06|3000.0|    High|       Medium|
|       7|       C007|    Mobile| 18000|  2024-01-07|1800.0|     Low|       Medium|
|       8|       C008|     Watch|  8000|  2024-01-07| 800.0|     Low|       

###Load the data

In [0]:
from pyspark.sql.functions import col

last_load_date = "2024-01-05"

incremental_df = df.filter(col("updated_date") > last_load_date)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window_spec = Window.partitionBy("order_id").orderBy(col("updated_date").desc())

dedup_df = df.withColumn("rn", row_number().over(window_spec)) \
             .filter(col("rn") == 1) \
             .drop("rn")